<a href="https://colab.research.google.com/github/riofutabac/PlacasVideos/blob/main/GoogleColab_ALPR_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Pipeline ALPR Orientado a Eventos — Vía de Lastre (Píntag)

**Arquitectura:** Crop físico + Motion Gate de 3 estados + YOLOv8 + ByteTrack + Top-M/Top-K + FastPlateOCR + Deduplicación + Reporte Excel con fotos incrustadas.

Repositorio: [`riofutabac/PlacasVideos`](https://github.com/riofutabac/PlacasVideos) (rama `main`).

> **Entorno de ejecución:** antes de correr cualquier celda, ve a `Entorno de ejecución → Cambiar tipo de entorno de ejecución` y selecciona **GPU (T4)**. Sin GPU, el pipeline funcionará pero mucho más lento.

## 1. Verificar GPU disponible

In [1]:
# Comprobar que Colab asignó una GPU NVIDIA (T4) y que PyTorch la detecta
!nvidia-smi

import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No se detectó GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4).")


Fri Sep 18 19:46:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Montar Google Drive

In [2]:
# Los videos de la cámara viven en Google Drive, dentro de la carpeta "Cam PL"
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 3. Clonar o actualizar el repositorio

In [3]:
import os

REPO_URL = "https://github.com/riofutabac/PlacasVideos.git"
REPO_DIR = "/content/PlacasVideos"
BRANCH = "main"

if not os.path.exists(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print("El repositorio ya existe, actualizando con git pull...")
    !cd "$REPO_DIR" && git fetch origin && git checkout "$BRANCH" && git pull origin "$BRANCH"

%cd $REPO_DIR
!git status


Cloning into '/content/PlacasVideos'...
remote: Enumerating objects: 377, done.
remote: Counting objects: 100% (377/377), done.
remote: Compressing objects: 100% (244/244), done.
remote: Total 377 (delta 230), reused 256 (delta 118), pack-reused 0 (from 0)
Receiving objects: 100% (377/377), 14.37 MiB | 32.77 MiB/s, done.
Resolving deltas: 100% (230/230), done.
/content/PlacasVideos
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


## 4. Instalar dependencias

Instalamos las dependencias del `requirements.txt` más las que Colab necesita de forma
específica: `onnxruntime-gpu` (compatible con CUDA 12, en vez del `onnxruntime` de CPU),
`fast-alpr`, y `onnx` (lo pide `ultralytics` para exportar, y si no está preinstalado
pide reiniciar el entorno de ejecución a mitad de la corrida).

In [4]:
# Instalar dependencias base del proyecto
!pip install -q -r requirements.txt
!pip install -q fast-alpr onnx

# Evitar conflicto de CUDA 13 en Colab: usar onnxruntime-gpu compatible con CUDA 12
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu==1.22.0

# Fijar versión de supervision compatible con el pipeline
!pip install -q "supervision<0.31" "opencv-python-headless<5"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.0/385.0 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 20.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [5]:
# Verificar versiones instaladas de las librerías clave
import cv2
import onnxruntime as ort
import supervision as sv
import ultralytics

print(f"OpenCV: {cv2.__version__}")
print(f"ONNX Runtime: {ort.__version__} | providers: {ort.get_available_providers()}")
print(f"Supervision: {sv.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
OpenCV: 4.14.0
ONNX Runtime: 1.22.0 | providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Supervision: 0.30.4
Ultralytics: 8.4.155


## 5. Configuración

In [6]:
# Carpeta de Drive donde están los videos de la cámara
VIDEOS_DIR = "/content/drive/MyDrive/Cam PL"

# Clips específicos para la corrida de validación rápida (60 y 61 tienen ground truth)
CLIPS = [60, 61]

# Límite de videos para una corrida de producción parcial (None = sin límite)
LIMIT = None


## 6. Corrida de validación rápida (clips 60 y 61)

Estos dos clips tienen datos de referencia (*ground truth*). Al incluirlos, `main.py`
imprime automáticamente al final un bloque **GROUND TRUTH EVALUATION** con precisión,
recall y las placas que coinciden o no contra el ground truth — úsalo para confirmar
que el pipeline sigue funcionando correctamente después de cualquier cambio.

In [7]:
!python main.py "$VIDEOS_DIR" --clips {" ".join(map(str, CLIPS))}


PIPELINE ALPR LIGERO ORIENTADO A EVENTOS v1.9.0
Inicio de corrida: 2026-09-18 19:48:13
📹 Videos a procesar (2 archivos):
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(60).mp4
  - /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(61).mp4
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading FastALPR on providers=['CUDAExecutionProvider', 'CPUExecutionProvider'] (device='cuda')...
INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['CUDAExecutionProvider', 'CPUExecutionProvider'] provider(s)
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
⚡ [SSD Staging] Copiando Camara Placas 2_20260

## 7. Corrida de producción

Procesa toda la carpeta de videos, o usa `--limit` para procesar solo los primeros N
archivos (útil para pruebas intermedias antes de lanzar la carpeta completa).

In [ ]:
# Toda la carpeta:
!python main.py "$VIDEOS_DIR"

# O bien, limitar la cantidad de videos a procesar (descomenta y ajusta LIMIT en la celda de configuración):
# !python main.py "$VIDEOS_DIR" --limit 10


## 8. Descargar el reporte Excel de auditoría

In [ ]:
import os
from google.colab import files

report_path = "reports/reporte_auditoria.xlsx"
if os.path.exists(report_path):
    files.download(report_path)
    print(f"📥 Descargando {report_path}...")
else:
    print(f"⚠️ No se encontró {report_path}. Corre el pipeline primero (secciones 6 o 7).")


## 9. Benchmarks / Diagnóstico (opcional)

Las siguientes celdas **no son necesarias** para generar el reporte de auditoría.
Sirven para diagnosticar rendimiento y comparar backends de inferencia/decodificación.
Ejecuta solo la(s) que necesites.

### 9.1 Benchmark de runtime del detector de vehículos (requiere GPU)

In [12]:
# Compara ultralytics vs ONNX Runtime (I/O binding) vs TensorRT para el modelo YOLO
!python benchmarks/benchmark_vehicle_runtime.py --model yolov8n.onnx


BENCHMARK DE RUNTIMES DE VEHÍCULO YOLOv8n (Fase C2/C3)
🖼️ Evaluando 11 cuadros ROI (2300x1064) @ imgsz=416...
⏱️ Probando backend: Ultralytics YOLO (Default)...
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
⏱️ Probando backend: ONNX Runtime CUDA I/O Binding...
⏱️ Probando backend: ONNX Runtime TensorRT EP (FP16)...
⏱️ Probando backend: Native TensorRT Engine (.engine)...
📁 Resultados guardados en benchmarks/benchmark_vehicle_runtime.csv

Backend / Runtime                   | Pre (ms) | Inf (ms) | Post (ms) | p50 (ms) | FPS    | Speedup
----------------------------------------------------------------------------------------

### 9.2 Diagnóstico A/B de decodificadores de video (NVDEC vs OpenCV)

In [9]:
# Compara el decodificador NVDEC (GPU) contra OpenCV (CPU) sobre el pipeline completo
!python benchmarks/diagnose_ab_decoders.py --full-pipeline


🔍 Paso 1/2: Diagnóstico numérico exacto en timestamps críticos (25, 105, 109, 112, 116, 120s)...
DIAGNÓSTICO COMPARATIVO A/B EXACTO: OpenCV vs NVDEC
Video: /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(60).mp4

Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
Loading FastALPR on providers=['CUDAExecutionProvider', 'CPUExecutionProvider'] (device='cuda')...
INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['CUDAExecutionProvider', 'CPUExecutionProvider'] provider(s)
Loading yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.22.0 with CUDAExecutionProvider
[OPENCV] Extrayendo frames en timestamps [25.0, 105.0, 109.0, 112.0, 116.0, 120.0]...
ℹ️ [VideoDecoder

### 9.3 Benchmark puro de FFmpeg + NVDEC

In [10]:
!python benchmarks/benchmark_ffmpeg_nvdec.py


BENCHMARK DE PIPELINES FFMPEG NVDEC (Clip 60: 8223 frames, 330.5s)
Video: /content/drive/MyDrive/Cam PL/Camara Placas 2_20260909105651-20260909163038(60).mp4

Testing: 1. NVDEC CUDA -> hwdownload -> NV12 (null sink)...
Traceback (most recent call last):
  File "/content/PlacasVideos/benchmarks/benchmark_ffmpeg_nvdec.py", line 133, in <module>
    main()
    ~~~~^^
  File "/content/PlacasVideos/benchmarks/benchmark_ffmpeg_nvdec.py", line 50, in main
    run_bench("1. NVDEC CUDA -> hwdownload -> NV12 (null sink)", [
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        "ffmpeg", "-hide_banner", "-nostdin", "-loglevel", "error",
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
        "-f", "null", "-"
        ^^^^^^^^^^^^^^^^^
    ])
    ^^
  File "/content/PlacasVideos/benchmarks/benchmark_ffmpeg_nvdec.py", line 26, in run_bench
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
  File "/usr/

### 9.4 Resumen ejecutivo de la última corrida

In [ ]:
# Resumen con métricas clave y costo estimado de GPU en la nube
!python benchmarks/resumen_ejecutivo.py --cost-per-hour 0.35


## 10. Solución de problemas

- **"archivo dañado" / error al abrir un clip:** revisa el error real impreso *justo
  arriba* de ese mensaje en la salida de la celda — normalmente indica la causa
  concreta (códec no soportado, archivo incompleto en Drive, etc.), y el mensaje de
  "archivo dañado" es solo el resumen final.
- **Cambiaste dependencias (paso 4) y algo falla de forma rara:** reinicia el entorno
  de ejecución (`Entorno de ejecución → Reiniciar entorno de ejecución`) y vuelve a
  ejecutar desde el paso 2 en adelante. Reinstalar `onnxruntime-gpu` u `onnx` sin
  reiniciar puede dejar el proceso de Python con la versión vieja cargada en memoria.
- **`git pull` falla con conflictos:** el repo en `/content/PlacasVideos` puede tener
  cambios locales de una corrida anterior. Bórralo (`!rm -rf /content/PlacasVideos`) y
  vuelve a ejecutar el paso 3 para clonar limpio.
- **No aparece el bloque GROUND TRUTH EVALUATION:** solo se imprime cuando la corrida
  incluye los clips 60 y/o 61 (`--clips 60 61`), porque son los únicos con datos de
  referencia.

In [11]:
!git pull

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 8 (delta 5), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 8.87 KiB | 2.96 MiB/s, done.
From https://github.com/riofutabac/PlacasVideos
   a6636c4..2f5a776  main       -> origin/main
Updating a6636c4..2f5a776
Fast-forward
 benchmarks/benchmark_vehicle_runtime.py |  41 +++--
 src/vehicle_runtime.py                  |  91 ++++++++---
 tests/test_vehicle_runtime.py           | 274 +++++++++++++++++++++++++++++++-
 3 files changed, 373 insertions(+), 33 deletions(-)
